# Week 3: Bilingual Benchmark Construction

Project: Fari Vision-Language Benchmark

Goal: Build a bilingual English-Mandarin image benchmark for evaluating object recognition and safety-related situations.

This notebook will be used to:
1. Load and inspect the label taxonomy.
2. Organize image metadata into a dataset manifest.
3. Remove duplicate images.
4. Create training, validation, and test splits.
5. Generate image corruptions for robustness testing.
6. Check that the benchmark can be reproduced.

In [ ]:
# Cell 1
# Import the libraries we will use throughout Week 3

import os
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Week 3 benchmark notebook is ready!")

In [ ]:
# Cell 2: Load the bilingual label taxonomy

# Replace this URL with the RAW URL of your GitHub CSV file
taxonomy_url = "https://raw.githubusercontent.com/cecezhuo/ingen_vlm_bench_v1.0/refs/heads/main/data/label_taxonomy.csv"

# Read the CSV file
taxonomy_df = pd.read_csv(taxonomy_url)

# Display the first five rows
display(taxonomy_df.head())

In [ ]:
# Cell 3: Check the complete taxonomy

# Count the total number of classes
print("Total number of classes:", len(taxonomy_df))

# Count the classes in each category
print("\nClasses by category:")
print(taxonomy_df["category"].value_counts())

# Display all 36 taxonomy entries
display(taxonomy_df)

In [ ]:
# Cell 4: Remove any repeated header rows

taxonomy_df = taxonomy_df[
    taxonomy_df["class_id"] != "class_id"
].copy()

# Reset the row numbers
taxonomy_df = taxonomy_df.reset_index(drop=True)

# Check the updated number of classes
print("Total number of classes:", len(taxonomy_df))

print("\nClasses by category:")
print(taxonomy_df["category"].value_counts())

In [ ]:
# Cell 5: Validate the taxonomy

# Check for duplicate class IDs
print("Duplicate class IDs:", taxonomy_df["class_id"].duplicated().sum())

# Check for missing values
print("Missing values:", taxonomy_df.isnull().sum().sum())

# Check that all 36 IDs are unique
print("Unique class IDs:", taxonomy_df["class_id"].nunique())

# Check that all expected columns exist
print("\nColumn names:")
print(taxonomy_df.columns.tolist())

In [ ]:
# Cell 6: Save the corrected taxonomy CSV

taxonomy_df.to_csv(
    "label_taxonomy.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Corrected taxonomy CSV saved successfully!")

In [ ]:
# Cell 7: Download the corrected taxonomy

from google.colab import files

files.download("label_taxonomy.csv")

In [ ]:
# Cell 8: Reload the cleaned taxonomy from GitHub

taxonomy_df = pd.read_csv(taxonomy_url)

print("Total classes:", len(taxonomy_df))
print("Duplicate class IDs:", taxonomy_df["class_id"].duplicated().sum())
print("Missing values:", taxonomy_df.isnull().sum().sum())

print("\nClasses by category:")
print(taxonomy_df["category"].value_counts())

print("\nEnglish class names:")
print(taxonomy_df["english_name"].tolist())

In [ ]:
# Cell 9: Create the dataset manifest template

import os
import pandas as pd

# Create the data folder if it does not already exist
os.makedirs("data", exist_ok=True)

# Define the columns for our dataset manifest
manifest_columns = [
    "image_id",
    "image_path",
    "source",
    "source_image_id",
    "license",
    "author",
    "class_id",
    "lighting",
    "clutter",
    "viewpoint",
    "split",
    "notes"
]

# Create an empty manifest with these columns
manifest_df = pd.DataFrame(columns=manifest_columns)

# Save the empty manifest
manifest_df.to_csv(
    "data/manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Manifest template created successfully!")
print("\nColumns:")
print(manifest_df.columns.tolist())

print("\nNumber of image records:", len(manifest_df))

In [ ]:
# Cell 10: Define manifest validation rules

valid_class_ids = set(taxonomy_df["class_id"])

valid_sources = {"Open Images", "Self-captured"}
valid_lighting = {"normal", "dim"}
valid_clutter = {"single", "cluttered"}
valid_viewpoints = {"web", "robot-height"}
valid_splits = {"adaptation", "validation", "test"}


def validate_manifest(df):
    errors = []

    # Check for missing required information
    required_columns = [
        "image_id", "image_path", "source", "license",
        "author", "class_id", "lighting", "clutter",
        "viewpoint", "split"
    ]

    for column in required_columns:
        if df[column].isnull().any():
            errors.append(f"Missing values in {column}")

    # Check that image IDs are unique
    if df["image_id"].duplicated().any():
        errors.append("Duplicate image IDs found")

    # Check that class IDs exist in our taxonomy
    invalid_classes = set(df["class_id"].dropna()) - valid_class_ids
    if invalid_classes:
        errors.append(f"Invalid class IDs: {invalid_classes}")

    # Check allowed values
    checks = {
        "source": valid_sources,
        "lighting": valid_lighting,
        "clutter": valid_clutter,
        "viewpoint": valid_viewpoints,
        "split": valid_splits
    }

    for column, allowed_values in checks.items():
        invalid_values = set(df[column].dropna()) - allowed_values
        if invalid_values:
            errors.append(f"Invalid {column} values: {invalid_values}")

    # Self-captured images cannot be in the adaptation split
    invalid_self_captured = df[
        (df["source"] == "Self-captured") &
        (df["split"] == "adaptation")
    ]

    if len(invalid_self_captured) > 0:
        errors.append("Self-captured images cannot be in the adaptation split")

    return errors


print("Manifest validation rules are ready!")

In [ ]:
# Cell 11: Test the manifest validation function

# Test 1: An empty manifest should have no errors
test_manifest = manifest_df.copy()

errors = validate_manifest(test_manifest)

print("Test 1: Empty manifest")
print("Errors:", errors)
print("Passed:", len(errors) == 0)

In [ ]:
# Cell 12: Test invalid manifest records

test_record = pd.DataFrame([{
    "image_id": "TEST001",
    "image_path": "images/test001.jpg",
    "source": "Self-captured",
    "source_image_id": "",
    "license": "N/A",
    "author": "Test",
    "class_id": "P01",
    "lighting": "normal",
    "clutter": "single",
    "viewpoint": "robot-height",
    "split": "adaptation",
    "notes": "This is an intentionally invalid test record"
}])

errors = validate_manifest(test_record)

print("Test 2: Invalid self-captured image")
print("Errors:", errors)
print("Passed:", any("Self-captured" in error for error in errors))

In [ ]:
# Cell 13: Test duplicate image IDs

duplicate_test = pd.DataFrame([
    {
        "image_id": "TEST002",
        "image_path": "images/test002.jpg",
        "source": "Open Images",
        "source_image_id": "abc123",
        "license": "CC BY 2.0",
        "author": "Test Author",
        "class_id": "P01",
        "lighting": "normal",
        "clutter": "single",
        "viewpoint": "web",
        "split": "validation",
        "notes": ""
    },
    {
        "image_id": "TEST002",
        "image_path": "images/test003.jpg",
        "source": "Open Images",
        "source_image_id": "def456",
        "license": "CC BY 2.0",
        "author": "Test Author",
        "class_id": "P02",
        "lighting": "dim",
        "clutter": "cluttered",
        "viewpoint": "web",
        "split": "test",
        "notes": ""
    }
])

errors = validate_manifest(duplicate_test)

print("Test 3: Duplicate image IDs")
print("Errors:", errors)
print("Passed:", "Duplicate image IDs found" in errors)

In [ ]:
# Cell 14: Test invalid class IDs

invalid_class_test = test_record.copy()

# Change the class ID to one that does not exist
invalid_class_test.loc[0, "class_id"] = "X99"

errors = validate_manifest(invalid_class_test)

print("Test 4: Invalid class ID")
print("Errors:", errors)
print("Passed:", any("Invalid class IDs" in error for error in errors))

In [ ]:
# Cell 15: Test missing values

missing_value_test = test_record.copy()

# Remove the author information
missing_value_test.loc[0, "author"] = None

errors = validate_manifest(missing_value_test)

print("Test 5: Missing required information")
print("Errors:", errors)
print("Passed:", any("Missing values in author" in error for error in errors))

In [ ]:
# Cell 16: Create folders for the benchmark images

folders = [
    "data/images/open_images",
    "data/images/self_captured",
    "data/images/processed",
    "data/splits",
    "data/corruptions"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Dataset folders created successfully!")

for folder in folders:
    print(folder)

In [ ]:
# Cell 17: Create a function for generating image IDs

def generate_image_id(source, number):
    if source == "Open Images":
        return f"OI_{number:05d}"
    elif source == "Self-captured":
        return f"SC_{number:05d}"
    else:
        raise ValueError("Unknown image source")


# Test the function
print("Open Images example:", generate_image_id("Open Images", 1))
print("Self-captured example:", generate_image_id("Self-captured", 1))

In [ ]:
# Cell 18: Create a function to add image records

def add_image_record(
    image_id,
    image_path,
    source,
    class_id,
    lighting,
    clutter,
    viewpoint,
    split,
    license,
    author,
    source_image_id="",
    notes=""
):
    global manifest_df

    new_record = {
        "image_id": image_id,
        "image_path": image_path,
        "source": source,
        "source_image_id": source_image_id,
        "license": license,
        "author": author,
        "class_id": class_id,
        "lighting": lighting,
        "clutter": clutter,
        "viewpoint": viewpoint,
        "split": split,
        "notes": notes
    }

    # Add the new record to the manifest
    manifest_df = pd.concat(
        [manifest_df, pd.DataFrame([new_record])],
        ignore_index=True
    )

    print(f"Added image record: {image_id}")


print("Image record function is ready!")

In [ ]:
# Cell 19: Test adding an image record

add_image_record(
    image_id="TEST001",
    image_path="data/images/open_images/test001.jpg",
    source="Open Images",
    class_id="P01",
    lighting="normal",
    clutter="single",
    viewpoint="web",
    split="validation",
    license="CC BY 2.0",
    author="Test Author",
    source_image_id="example123",
    notes="Temporary test record"
)

print("\nCurrent manifest:")
display(manifest_df)

In [ ]:
# Cell 20: Remove the temporary test record

manifest_df = manifest_df[
    manifest_df["image_id"] != "TEST001"
].copy()

manifest_df = manifest_df.reset_index(drop=True)

print("Temporary test record removed!")
print("Current number of image records:", len(manifest_df))

In [ ]:
# Cell 21: Final manifest validation check

errors = validate_manifest(manifest_df)

print("Manifest validation results:")
print("Number of image records:", len(manifest_df))
print("Number of errors:", len(errors))
print("Errors:", errors)

if len(errors) == 0:
    print("\nManifest is ready for image collection!")

In [ ]:
# Cell 22: Save the clean manifest

manifest_df.to_csv(
    "data/manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Manifest saved successfully!")
print("File location: data/manifest.csv")
print("Number of image records:", len(manifest_df))

In [ ]:
# Cell 23: Load the official Open Images V7 class list

open_images_url = (
    "https://storage.googleapis.com/openimages/v7/"
    "oidv7-class-descriptions-boxable.csv"
)

open_images_classes = pd.read_csv(
    open_images_url,
    header=None,
    names=["open_images_class_id", "open_images_name"]
)

print("Open Images V7 class list loaded!")
print("Total available classes:", len(open_images_classes))

display(open_images_classes.head(10))

In [ ]:
# Cell 24: Clean the Open Images class list

open_images_classes = open_images_classes[
    open_images_classes["open_images_class_id"] != "LabelName"
].copy()

open_images_classes = open_images_classes.reset_index(drop=True)

print("Open Images classes after cleaning:", len(open_images_classes))
print("\nFirst 10 classes:")
display(open_images_classes.head(10))

In [ ]:
# Cell 25: Compare taxonomy labels with Open Images classes

open_images_names = set(
    open_images_classes["open_images_name"].str.lower().str.strip()
)

taxonomy_df["open_images_match"] = (
    taxonomy_df["english_name"].str.lower().str.strip().isin(open_images_names)
)

print("Direct matches:",
      taxonomy_df["open_images_match"].sum())

print("No direct match:",
      (~taxonomy_df["open_images_match"]).sum())

display(
    taxonomy_df[
        ["class_id", "english_name", "open_images_match"]
    ]
)

In [ ]:
# Cell 26: Search Open Images for alternative labels

unmatched_classes = taxonomy_df[
    ~taxonomy_df["open_images_match"]
]

search_terms = [
    "key", "phone", "mobile", "wallet", "medicine",
    "pill", "glass", "water bottle", "bottle",
    "container", "sofa", "couch", "cabinet",
    "floor", "liquid", "spill", "door", "walkway",
    "obstacle", "fallen"
]

for term in search_terms:
    matches = open_images_classes[
        open_images_classes["open_images_name"].str.contains(
            term, case=False, na=False
        )
    ]

    if not matches.empty:
        print(f"\nSearch term: {term}")
        display(matches)

In [ ]:
# Cell 27: Create a manual Open Images matching table

class_matches = {
    "P01": ("Keys", None, "No suitable match found"),
    "P03": ("Phone", "Mobile phone", "Good match"),
    "P04": ("Wallet", None, "No suitable match found"),
    "P05": ("Medicine bottle", "Bottle", "Too broad"),
    "K02": ("Drinking glass", "Wine glass", "Partial match"),
    "K05": ("Water bottle", "Bottle", "Too broad"),
    "K10": ("Food container", "Container", "Too broad"),
    "F02": ("Sofa", "Couch", "Good match"),
    "F08": ("Cabinet", "Cabinetry", "Needs review"),
    "S01": ("Object on floor", None, "No suitable match found"),
    "S02": ("Spilled liquid", None, "No suitable match found"),
    "S03": ("Open door", None, "No suitable match found"),
    "S04": ("Obstructed walkway", None, "No suitable match found"),
    "S05": ("Open cabinet", None, "No suitable match found"),
    "S06": ("Fallen object", None, "No suitable match found")
}

matching_df = pd.DataFrame([
    {
        "class_id": class_id,
        "taxonomy_label": values[0],
        "open_images_candidate": values[1],
        "assessment": values[2]
    }
    for class_id, values in class_matches.items()
])

display(matching_df)

In [ ]:
# Cell 28: Save the Open Images matching table

matching_df.to_csv(
    "data/open_images_class_matches.csv",
    index=False
)

print("Matching table saved!")
print("Total reviewed classes:", len(matching_df))

In [ ]:
# Cell 29: Display all 36 taxonomy classes and their Open Images matches

display(
    taxonomy_df[
        ["class_id", "english_name", "open_images_match"]
    ].sort_values("class_id")
)

In [ ]:
# Cell 30: Create a list of exact Open Images matches

exact_matches = taxonomy_df[
    taxonomy_df["open_images_match"] == True
][["class_id", "english_name"]].copy()

print("Number of exact matches:", len(exact_matches))
print("\nClasses available for Open Images sourcing:")

display(exact_matches.sort_values("class_id"))

In [ ]:
# Cell 31: Create a lookup table for our 21 Open Images categories

open_images_lookup = exact_matches.merge(
    open_images_classes,
    left_on="english_name",
    right_on="open_images_name",
    how="left"
)

display(
    open_images_lookup[
        ["class_id", "english_name", "open_images_class_id"]
    ].sort_values("class_id")
)

print("Categories with Open Images IDs:",
      open_images_lookup["open_images_class_id"].notna().sum())

In [ ]:
# Cell 32: Load a sample of Open Images V7

!pip -q install fiftyone

import fiftyone as fo
import fiftyone.zoo as foz

classes = exact_matches["english_name"].tolist()

open_images_sample = foz.load_zoo_dataset(
    "open-images-v7",
    split="validation",
    label_types=["detections"],
    classes=classes,
    max_samples=100
)

print("Images downloaded:", len(open_images_sample))

In [ ]:
# Cell 33: Count the objects found in our downloaded images

object_counts = {}

for sample in open_images_sample:
    for detection in sample.ground_truth.detections:
        label = detection.label
        object_counts[label] = object_counts.get(label, 0) + 1

counts_df = pd.DataFrame(
    object_counts.items(),
    columns=["object", "count"]
).sort_values("count", ascending=False)

display(counts_df)

In [ ]:
# Cell 34: Check counts for our 21 target categories

target_counts = exact_matches.merge(
    counts_df,
    left_on="english_name",
    right_on="object",
    how="left"
)

target_counts["count"] = target_counts["count"].fillna(0).astype(int)

display(
    target_counts[["class_id", "english_name", "count"]]
    .sort_values("class_id")
)

In [ ]:
# Cell 35: Count images containing each object

image_counts = {}

for sample in open_images_sample:
    labels = set(d.label for d in sample.ground_truth.detections)

    for label in labels:
        image_counts[label] = image_counts.get(label, 0) + 1

target_counts["image_count"] = (
    target_counts["english_name"]
    .map(image_counts)
    .fillna(0)
    .astype(int)
)

display(
    target_counts[["class_id", "english_name", "image_count"]]
    .sort_values("class_id")
)

In [ ]:
# Cell 36: Download up to 30 images for each category

class_datasets = {}

for label in exact_matches["english_name"]:
    name = label.lower().replace(" ", "-")

    dataset = foz.load_zoo_dataset(
        "open-images-v7",
        split="validation",
        label_types=["detections"],
        classes=[label],
        max_samples=30,
        shuffle=True,
        seed=42,
        name=f"oi-{name}"
    )

    class_datasets[label] = dataset
    print(label, ":", len(dataset), "images")

In [ ]:
# Cell 37: Check the downloaded category datasets

for label, dataset in class_datasets.items():
    print(label, ":", len(dataset), "images")

In [ ]:
# Cell 37: Reload each category with its own dataset name

class_datasets = {}

for label in exact_matches["english_name"]:
    name = label.lower().replace(" ", "-")

    dataset = foz.load_zoo_dataset(
        "open-images-v7",
        split="validation",
        label_types=["detections"],
        classes=[label],
        max_samples=30,
        shuffle=True,
        seed=42,
        dataset_name=f"oi-{name}-validation"
    )

    class_datasets[label] = dataset
    print(label, ":", len(dataset), "images")

In [ ]:
# Cell 38: Verify separate category datasets

for label, dataset in class_datasets.items():
    print(
        label,
        "| Dataset:", dataset.name,
        "| Images:", len(dataset)
    )

print("\nNumber of separate datasets:", len(set(
    dataset.name for dataset in class_datasets.values()
)))

In [ ]:
# Cell 39: Display sample Glasses images

glasses_dataset = class_datasets["Glasses"]

samples = glasses_dataset.take(12, seed=42)

session = fo.launch_app(samples)

In [ ]:
# Cell 40: Inspect Glasses dataset information

glasses_dataset = class_datasets["Glasses"]

print("Dataset fields:")
print(glasses_dataset.get_field_schema())

print("\nFirst image information:")
print(glasses_dataset.first().to_dict())

In [ ]:
# Cell 41: Inspect image IDs and object labels

for sample in glasses_dataset.limit(5):
    print("\nImage:", sample.filepath.split("/")[-1])
    print("Labels:", [
        detection.label
        for detection in sample.ground_truth.detections
    ])

In [ ]:
# Cell 42: Load Glasses images with original Open Images IDs

glasses_dataset = foz.load_zoo_dataset(
    "open-images-v7",
    split="validation",
    label_types=["detections"],
    classes=["Glasses"],
    max_samples=30,
    shuffle=True,
    seed=42,
    include_id=True,
    dataset_name="oi-glasses-validation-with-ids"
)

print("Number of images:", len(glasses_dataset))
print("\nDataset fields:")
print(glasses_dataset.get_field_schema())

In [ ]:
# Cell 43: Extract Open Images IDs from filenames

glasses_records = []

for sample in glasses_dataset:
    image_id = sample.filepath.split("/")[-1].replace(".jpg", "")

    glasses_records.append({
        "image_id": image_id,
        "image_path": sample.filepath,
        "class_id": "P02",
        "source": "Open Images"
    })

glasses_preview = pd.DataFrame(glasses_records)

print("Number of records:", len(glasses_preview))
display(glasses_preview.head())

In [ ]:
# Cell 44: Download official validation image metadata

metadata_url = (
    "https://storage.googleapis.com/openimages/v7/oidv7-"
    "validation-annotations-bbox.csv"
)

print("Checking available metadata files...")

In [ ]:
# Cell 45: Download Open Images validation metadata

metadata_url = (
    "https://storage.googleapis.com/openimages/2018_04/"
    "validation/validation-images-with-rotation.csv"
)

metadata_df = pd.read_csv(metadata_url)

print("Metadata rows:", len(metadata_df))
print("Columns:", metadata_df.columns.tolist())

display(metadata_df.head())

In [ ]:
# Cell 46: Match Glasses images with official metadata

glasses_metadata = glasses_preview.merge(
    metadata_df,
    left_on="image_id",
    right_on="ImageID",
    how="left"
)

print("Matched images:", glasses_metadata["License"].notna().sum())
print("Missing metadata:", glasses_metadata["License"].isna().sum())

display(
    glasses_metadata[
        ["image_id", "class_id", "Author", "License", "OriginalLandingURL"]
    ].head(10)
)

In [ ]:
# Cell 47: Collect candidate images across all categories

class_datasets["Glasses"] = glasses_dataset

all_records = []

for label, dataset in class_datasets.items():
    class_id = exact_matches.loc[
        exact_matches["english_name"] == label, "class_id"
    ].iloc[0]

    for sample in dataset:
        image_id = sample.filepath.split("/")[-1].replace(".jpg", "")

        all_records.append({
            "image_id": image_id,
            "image_path": sample.filepath,
            "class_id": class_id,
            "english_name": label,
            "source": "Open Images"
        })

all_candidates_df = pd.DataFrame(all_records)

all_candidates_df = all_candidates_df.merge(
    metadata_df,
    left_on="image_id",
    right_on="ImageID",
    how="left"
)

print("Total candidate records:", len(all_candidates_df))
print("Missing license records:", all_candidates_df["License"].isna().sum())
print("Unique image IDs:", all_candidates_df["image_id"].nunique())
print("Duplicate image IDs:", all_candidates_df["image_id"].duplicated().sum())

display(
    all_candidates_df.groupby("english_name").size().reset_index(name="images")
)

In [ ]:
# Cell 48: Identify images appearing in multiple categories

image_category_counts = (
    all_candidates_df.groupby("image_id")["class_id"]
    .nunique()
    .reset_index(name="number_of_categories")
)

multi_category_images = image_category_counts[
    image_category_counts["number_of_categories"] > 1
]

print("Images with multiple categories:", len(multi_category_images))

display(
    all_candidates_df[
        all_candidates_df["image_id"].isin(
            multi_category_images["image_id"]
        )
    ][["image_id", "english_name", "class_id"]]
    .sort_values("image_id")
    .head(30)
)

In [ ]:
# Cell 49: Combine categories for each unique image

unique_images_df = (
    all_candidates_df.groupby("image_id")
    .agg({
        "image_path": "first",
        "source": "first",
        "License": "first",
        "Author": "first",
        "OriginalLandingURL": "first",
        "english_name": lambda x: sorted(set(x)),
        "class_id": lambda x: sorted(set(x))
    })
    .reset_index()
)

unique_images_df["number_of_categories"] = (
    unique_images_df["class_id"].apply(len)
)

print("Unique images:", len(unique_images_df))
print("Images with multiple categories:",
      (unique_images_df["number_of_categories"] > 1).sum())

display(unique_images_df.head(10))

In [ ]:
# Cell 50: Count unique images per category

category_counts = (
    unique_images_df.explode("class_id")
    .groupby("class_id")["image_id"]
    .nunique()
    .reset_index(name="unique_images")
)

category_counts = category_counts.merge(
    exact_matches[["class_id", "english_name"]],
    on="class_id",
    how="right"
).fillna({"unique_images": 0})

category_counts["unique_images"] = (
    category_counts["unique_images"].astype(int)
)

display(category_counts.sort_values("class_id"))

print("\nTotal unique images:", unique_images_df["image_id"].nunique())
print("Categories represented:", (category_counts["unique_images"] > 0).sum())
print("Categories below 30 images:", (category_counts["unique_images"] < 30).sum())

In [ ]:
# Cell 51: Calculate image gaps

category_counts["images_needed"] = (
    30 - category_counts["unique_images"]
).clip(lower=0)

print("Additional images needed:")
display(
    category_counts[category_counts["images_needed"] > 0][
        ["class_id", "english_name", "unique_images", "images_needed"]
    ]
)

print(
    "\nTotal additional images needed:",
    category_counts["images_needed"].sum()
)

In [ ]:
# Cell 52: Download additional candidates from Open Images training

additional_datasets = {}

for label in ["Remote control", "Backpack", "Watch"]:
    name = label.lower().replace(" ", "-")

    dataset = foz.load_zoo_dataset(
        "open-images-v7",
        split="train",
        label_types=["detections"],
        classes=[label],
        max_samples=50,
        shuffle=True,
        seed=42,
        dataset_name=f"oi-{name}-train"
    )

    additional_datasets[label] = dataset

    print(label, ":", len(dataset), "additional candidates")

In [ ]:
# Cell 53: Check which project files survived

import os

for folder in ["/content/data", "/content/data/images/open_images"]:
    print("\n", folder)
    if os.path.exists(folder):
        print(os.listdir(folder)[:20])
    else:
        print("Folder not found")

In [ ]:
# Cell 54: Reload saved CSV files

import pandas as pd
import os

taxonomy_df = pd.read_csv(
    "/content/data/label_taxonomy.csv"
)

matching_df = pd.read_csv(
    "/content/data/open_images_class_matches.csv"
)

manifest_df = pd.read_csv(
    "/content/data/manifest.csv"
)

print("Taxonomy:", len(taxonomy_df), "categories")
print("Class matches:", len(matching_df), "records")
print("Manifest:", len(manifest_df), "records")

In [ ]:
# Cell 55: Check saved files and reload available data

import os
import pandas as pd

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

matching_df = pd.read_csv(
    "/content/data/open_images_class_matches.csv"
)

manifest_df = pd.read_csv(
    "/content/data/manifest.csv"
)

print("\nClass matches:", len(matching_df))
print("Manifest records:", len(manifest_df))

In [ ]:
# Cell 56: Reload taxonomy from its actual location

taxonomy_df = pd.read_csv("/content/label_taxonomy.csv")

print("Taxonomy loaded:", len(taxonomy_df), "categories")
print("Missing values:", taxonomy_df.isnull().sum().sum())
display(taxonomy_df.head())

In [ ]:
# Cell 57: Check available FiftyOne datasets

import fiftyone as fo

print("Available FiftyOne datasets:")

for dataset in fo.list_datasets():
    print(dataset)

In [ ]:
# Cell 58: Check recovered image counts

for name in fo.list_datasets():
    if name.startswith("oi-"):
        dataset = fo.load_dataset(name)
        print(name, ":", len(dataset), "images")

In [ ]:
# Cell 59: Rebuild candidate records from recovered datasets

import os
import pandas as pd
import fiftyone as fo

# Match dataset names to our taxonomy
taxonomy_lookup = {
    name.lower(): (class_id, name)
    for class_id, name in zip(
        taxonomy_df["class_id"],
        taxonomy_df["english_name"]
    )
}

all_records = []

for dataset_name in fo.list_datasets():

    if not dataset_name.startswith("oi-"):
        continue

    # Skip the duplicate Glasses dataset
    if dataset_name == "oi-glasses-validation-with-ids":
        continue

    parts = dataset_name[3:].rsplit("-", 1)

    if len(parts) != 2 or parts[1] not in ["validation", "train"]:
        continue

    label = parts[0].replace("-", " ").lower()
    open_images_split = parts[1]

    if label not in taxonomy_lookup:
        continue

    class_id, english_name = taxonomy_lookup[label]
    dataset = fo.load_dataset(dataset_name)

    for sample in dataset:
        image_id = os.path.basename(sample.filepath).replace(".jpg", "")

        all_records.append({
            "image_id": image_id,
            "image_path": sample.filepath,
            "class_id": class_id,
            "english_name": english_name,
            "source": "Open Images",
            "open_images_split": open_images_split
        })

all_candidates_df = pd.DataFrame(all_records)

print("Candidate records:", len(all_candidates_df))
print("Unique images:", all_candidates_df["image_id"].nunique())
print("Categories:", all_candidates_df["class_id"].nunique())
print("Image files still available:",
      all_candidates_df["image_path"].apply(os.path.exists).sum())

In [ ]:
# Cell 60: Reload Open Images licensing metadata

metadata_url = (
    "https://storage.googleapis.com/openimages/2018_04/"
    "validation/validation-images-with-rotation.csv"
)

metadata_df = pd.read_csv(metadata_url)

print("Metadata rows:", len(metadata_df))
print("Available columns:")
print(metadata_df.columns.tolist())

In [ ]:
# Cell 61: Add official licensing and attribution information

all_candidates_df = all_candidates_df.merge(
    metadata_df,
    left_on="image_id",
    right_on="ImageID",
    how="left"
)

print("Total candidate records:", len(all_candidates_df))
print("Records with license:", all_candidates_df["License"].notna().sum())
print("Records missing license:", all_candidates_df["License"].isna().sum())

display(
    all_candidates_df[
        ["image_id", "english_name", "License", "Author", "OriginalLandingURL"]
    ].head(10)
)

In [ ]:
# Cell 62: Load Open Images training metadata

train_metadata_url = (
    "https://storage.googleapis.com/openimages/2018_04/"
    "train/train-images-boxable-with-rotation.csv"
)

train_metadata_df = pd.read_csv(train_metadata_url)

print("Training metadata rows:", len(train_metadata_df))

train_ids = all_candidates_df.loc[
    all_candidates_df["open_images_split"] == "train",
    "image_id"
].unique()

matched_train_ids = train_metadata_df[
    train_metadata_df["ImageID"].isin(train_ids)
]

print("Training image IDs to match:", len(train_ids))
print("Training image IDs matched:", len(matched_train_ids))

In [ ]:
# Cell 63: Merge training metadata into candidate records

train_columns = [
    "ImageID", "OriginalURL", "OriginalLandingURL",
    "License", "AuthorProfileURL", "Author", "Title",
    "OriginalSize", "OriginalMD5", "Thumbnail300KURL", "Rotation"
]

train_metadata_matched = train_metadata_df[
    train_metadata_df["ImageID"].isin(train_ids)
][train_columns]

# Remove the old incomplete metadata columns
metadata_columns = [
    "ImageID", "Subset", "OriginalURL", "OriginalLandingURL",
    "License", "AuthorProfileURL", "Author", "Title",
    "OriginalSize", "OriginalMD5", "Thumbnail300KURL", "Rotation"
]

all_candidates_df = all_candidates_df.drop(
    columns=[col for col in metadata_columns if col in all_candidates_df.columns]
)

# Attach the correct metadata for both splits
all_candidates_df = all_candidates_df.merge(
    pd.concat([
        metadata_df[metadata_columns],
        train_metadata_matched
    ]).drop_duplicates(subset="ImageID"),
    left_on="image_id",
    right_on="ImageID",
    how="left"
)

print("Total candidate records:", len(all_candidates_df))
print("Records with license:", all_candidates_df["License"].notna().sum())
print("Records missing license:", all_candidates_df["License"].isna().sum())
print("Records missing author:", all_candidates_df["Author"].isna().sum())

In [ ]:
# Cell 64: Combine records for each unique image

unique_images_df = (
    all_candidates_df
    .groupby("image_id")
    .agg({
        "image_path": "first",
        "source": "first",
        "open_images_split": "first",
        "License": "first",
        "Author": "first",
        "OriginalLandingURL": "first",
        "class_id": lambda x: sorted(set(x)),
        "english_name": lambda x: sorted(set(x))
    })
    .reset_index()
)

print("Original candidate records:", len(all_candidates_df))
print("Unique images:", len(unique_images_df))
print(
    "Images with multiple categories:",
    (unique_images_df["class_id"].apply(len) > 1).sum()
)

display(unique_images_df.head(10))

In [ ]:
# Cell 65: Check exact duplicate image files

import hashlib
from collections import defaultdict

hash_groups = defaultdict(list)

for _, row in unique_images_df.iterrows():
    image_path = row["image_path"]

    with open(image_path, "rb") as image_file:
        file_hash = hashlib.md5(image_file.read()).hexdigest()

    hash_groups[file_hash].append(row["image_id"])

duplicate_groups = [
    ids for ids in hash_groups.values() if len(ids) > 1
]

print("Unique images checked:", len(unique_images_df))
print("Exact duplicate groups:", len(duplicate_groups))
print(
    "Images involved in exact duplicates:",
    sum(len(group) for group in duplicate_groups)
)

In [ ]:
# Cell 66: Check visually similar images using perceptual hashing

!pip -q install ImageHash

import imagehash
from PIL import Image
from itertools import combinations

image_hashes = {}

for _, row in unique_images_df.iterrows():
    image = Image.open(row["image_path"]).convert("RGB")
    image_hashes[row["image_id"]] = imagehash.phash(image)

similar_pairs = []

for (id1, hash1), (id2, hash2) in combinations(image_hashes.items(), 2):
    distance = hash1 - hash2

    if distance <= 5:
        similar_pairs.append({
            "image_id_1": id1,
            "image_id_2": id2,
            "hash_distance": distance
        })

similar_df = pd.DataFrame(similar_pairs)

print("Images hashed:", len(image_hashes))
print("Potentially similar pairs:", len(similar_df))

display(similar_df.head(20))

In [ ]:
# Cell 67: Visually inspect the potentially similar pair

from IPython.display import display

similar_ids = ["9465a2acc11bce7d", "98088cd2e9d35d41"]

for image_id in similar_ids:
    row = unique_images_df[
        unique_images_df["image_id"] == image_id
    ].iloc[0]

    print("Image ID:", image_id)
    print("Categories:", row["english_name"])
    display(Image.open(row["image_path"]).convert("RGB"))

In [ ]:
# Cell 68: Record and exclude the near-duplicate

near_duplicate_review = pd.DataFrame([{
    "image_id_1": "9465a2acc11bce7d",
    "image_id_2": "98088cd2e9d35d41",
    "decision": "Keep image 1; exclude image 2",
    "reason": "Near-identical JVC car stereo product images"
}])

near_duplicate_review.to_csv(
    "/content/data/near_duplicate_review.csv",
    index=False
)

excluded_image_ids = ["98088cd2e9d35d41"]

benchmark_images_df = unique_images_df[
    ~unique_images_df["image_id"].isin(excluded_image_ids)
].copy()

print("Images before review:", len(unique_images_df))
print("Images excluded:", len(excluded_image_ids))
print("Images after review:", len(benchmark_images_df))

In [ ]:
# Cell 69: Check how many images remain in each category

category_counts = {}

for _, row in benchmark_images_df.iterrows():
    for category in row["english_name"]:
        category_counts[category] = category_counts.get(category, 0) + 1

category_counts_df = pd.DataFrame(
    list(category_counts.items()),
    columns=["Category", "Image Count"]
).sort_values("Category")

display(category_counts_df)

print("Total unique images:", len(benchmark_images_df))
print("Categories represented:", len(category_counts_df))
print("Categories below 30 images:",
      (category_counts_df["Image Count"] < 30).sum())

In [ ]:
# Cell 70: Check image counts by original Open Images split

split_counts = benchmark_images_df.groupby(
    "open_images_split"
).size().reset_index(name="Image Count")

display(split_counts)

print("Images by category and original split:")

category_split_counts = (
    benchmark_images_df
    .explode("english_name")
    .groupby(["english_name", "open_images_split"])
    .size()
    .unstack(fill_value=0)
)

display(category_split_counts)

In [ ]:
# Cell 71: Verify that original Open Images splits are preserved

print("Adaptation candidates:")
print(
    len(benchmark_images_df[
        benchmark_images_df["open_images_split"] == "train"
    ])
)

print("\nValidation/test candidates:")
print(
    len(benchmark_images_df[
        benchmark_images_df["open_images_split"] == "validation"
    ])
)

print("\nCheck for unexpected split values:")
print(
    benchmark_images_df["open_images_split"]
    .value_counts()
)

In [ ]:
# Cell 72: Build the benchmark manifest

manifest_df = benchmark_images_df[[
    "image_id",
    "image_path",
    "source",
    "License",
    "Author",
    "OriginalLandingURL",
    "class_id",
    "english_name",
    "open_images_split"
]].copy()

manifest_df = manifest_df.rename(columns={
    "License": "license",
    "Author": "author",
    "OriginalLandingURL": "source_url",
    "class_id": "class_ids",
    "english_name": "english_labels"
})

# Store multiple labels as semicolon-separated values
manifest_df["class_ids"] = manifest_df["class_ids"].apply(
    lambda labels: ";".join(labels)
)

manifest_df["english_labels"] = manifest_df["english_labels"].apply(
    lambda labels: ";".join(labels)
)

# Add fields that we will fill during benchmark construction
manifest_df["lighting"] = "unknown"
manifest_df["clutter"] = "unknown"
manifest_df["viewpoint"] = "unknown"
manifest_df["benchmark_split"] = "unassigned"

manifest_df.to_csv(
    "/content/data/manifest.csv",
    index=False
)

print("Manifest rows:", len(manifest_df))
print("Manifest columns:", list(manifest_df.columns))
display(manifest_df.head())

In [ ]:
# Cell 73: Copy images into the benchmark folder

import os
import shutil

benchmark_image_folder = "/content/data/images/open_images"
os.makedirs(benchmark_image_folder, exist_ok=True)

copied = 0

for _, row in manifest_df.iterrows():
    source_path = row["image_path"]
    destination_path = os.path.join(
        benchmark_image_folder,
        row["image_id"] + ".jpg"
    )

    if not os.path.exists(destination_path):
        shutil.copy2(source_path, destination_path)
        copied += 1

    manifest_df.loc[
        manifest_df["image_id"] == row["image_id"],
        "image_path"
    ] = "data/images/open_images/" + row["image_id"] + ".jpg"

print("Images copied:", copied)
print("Images in benchmark folder:", len(os.listdir(benchmark_image_folder)))
print("Example repository path:", manifest_df.iloc[0]["image_path"])

In [ ]:
# Cell 74: Save and verify the manifest

manifest_df.to_csv("/content/data/manifest.csv", index=False)

missing_images = []

for image_path in manifest_df["image_path"]:
    full_path = "/content/" + image_path

    if not os.path.exists(full_path):
        missing_images.append(image_path)

print("Manifest rows:", len(manifest_df))
print("Missing image files:", len(missing_images))
print("Duplicate image IDs:", manifest_df["image_id"].duplicated().sum())
print("Missing licenses:", manifest_df["license"].isna().sum())
print("Missing authors:", manifest_df["author"].isna().sum())

In [ ]:
# Cell 75: Create reproducible benchmark splits

import numpy as np

rng = np.random.default_rng(42)

manifest_df["benchmark_split"] = "unassigned"

# Original Open Images training images go to adaptation
manifest_df.loc[
    manifest_df["open_images_split"] == "train",
    "benchmark_split"
] = "adaptation"

# Divide validation images into validation and test
validation_indices = manifest_df.index[
    manifest_df["open_images_split"] == "validation"
].to_list()

rng.shuffle(validation_indices)

midpoint = len(validation_indices) // 2

manifest_df.loc[validation_indices[:midpoint], "benchmark_split"] = "validation"
manifest_df.loc[validation_indices[midpoint:], "benchmark_split"] = "test"

print(manifest_df["benchmark_split"].value_counts())
print("\nOriginal source vs benchmark split:")
display(pd.crosstab(
    manifest_df["open_images_split"],
    manifest_df["benchmark_split"]
))

In [ ]:
# Cell 76: Check category distribution across benchmark splits

split_category_counts = (
    manifest_df.assign(
        category=manifest_df["english_labels"].str.split(";")
    )
    .explode("category")
    .groupby(["category", "benchmark_split"])
    .size()
    .unstack(fill_value=0)
)

display(split_category_counts)

print("\nTotal categories:", len(split_category_counts))
print("\nCategories with zero test images:")
print(
    split_category_counts[
        split_category_counts["test"] == 0
    ].index.tolist()
)

In [ ]:
# Cell 77: Review Remote control image assignments

remote_control_rows = manifest_df[
    manifest_df["english_labels"].str.contains("Remote control")
][[
    "image_id",
    "class_ids",
    "english_labels",
    "open_images_split",
    "benchmark_split"
]].sort_values("benchmark_split")

display(remote_control_rows)

print("\nRemote control counts:")
print(remote_control_rows["benchmark_split"].value_counts())

In [ ]:
# Cell 78: Reassign Remote control validation/test images

remote_validation_ids = remote_control_rows[
    remote_control_rows["open_images_split"] == "validation"
]["image_id"].tolist()

# Keep the first 3 for validation and the remaining 3 for test
manifest_df.loc[
    manifest_df["image_id"].isin(remote_validation_ids[:3]),
    "benchmark_split"
] = "validation"

manifest_df.loc[
    manifest_df["image_id"].isin(remote_validation_ids[3:]),
    "benchmark_split"
] = "test"

# Check the updated counts
remote_updated = manifest_df[
    manifest_df["english_labels"].str.contains("Remote control")
]

print(remote_updated["benchmark_split"].value_counts())

In [ ]:
# Cell 79: Recheck category distribution after adjustment

split_category_counts = (
    manifest_df.assign(
        category=manifest_df["english_labels"].str.split(";")
    )
    .explode("category")
    .groupby(["category", "benchmark_split"])
    .size()
    .unstack(fill_value=0)
)

display(split_category_counts)

print("\nTotal images:", len(manifest_df))
print("\nOverall split counts:")
print(manifest_df["benchmark_split"].value_counts())

In [ ]:
# Cell 80: Save final split assignments

manifest_df.to_csv("/content/data/manifest.csv", index=False)

split_summary = manifest_df["benchmark_split"].value_counts().reset_index()
split_summary.columns = ["benchmark_split", "image_count"]

split_summary.to_csv(
    "/content/data/split_summary.csv",
    index=False
)

print("Manifest saved.")
print("Split summary saved.")
display(split_summary)

In [ ]:
# Cell 81: Set up image corruption tools

!pip -q install opencv-python-headless

import cv2
import numpy as np
from PIL import Image
import os

corruption_folder = "/content/data/corruptions"
os.makedirs(corruption_folder, exist_ok=True)

print("OpenCV version:", cv2.__version__)
print("Corruption output folder:", corruption_folder)

In [ ]:
# Cell 82: Define image corruption functions

def apply_corruption(image, corruption, severity, seed=42):
    image = np.array(image).astype(np.uint8)
    rng = np.random.default_rng(seed)

    if corruption == "low_brightness":
        factors = {1: 0.7, 2: 0.5, 3: 0.3}
        result = image * factors[severity]

    elif corruption == "gaussian_noise":
        strengths = {1: 10, 2: 20, 3: 30}
        noise = rng.normal(0, strengths[severity], image.shape)
        result = image.astype(float) + noise

    elif corruption == "motion_blur":
        sizes = {1: 5, 2: 9, 3: 15}
        size = sizes[severity]
        kernel = np.zeros((size, size))
        kernel[size // 2, :] = 1 / size
        result = cv2.filter2D(image, -1, kernel)

    elif corruption == "defocus_blur":
        sizes = {1: 3, 2: 7, 3: 11}
        size = sizes[severity]
        kernel = np.zeros((size, size), dtype=np.float32)
        center = size // 2
        for y in range(size):
            for x in range(size):
                if (x - center) ** 2 + (y - center) ** 2 <= center ** 2:
                    kernel[y, x] = 1
        kernel /= kernel.sum()
        result = cv2.filter2D(image, -1, kernel)

    elif corruption == "jpeg_compression":
        qualities = {1: 70, 2: 40, 3: 15}
        success, encoded = cv2.imencode(
            ".jpg", cv2.cvtColor(image, cv2.COLOR_RGB2BGR),
            [cv2.IMWRITE_JPEG_QUALITY, qualities[severity]]
        )
        result = cv2.cvtColor(
            cv2.imdecode(encoded, cv2.IMREAD_COLOR),
            cv2.COLOR_BGR2RGB
        )

    else:
        raise ValueError("Unknown corruption type")

    return np.clip(result, 0, 255).astype(np.uint8)


corruption_types = [
    "low_brightness",
    "gaussian_noise",
    "motion_blur",
    "defocus_blur",
    "jpeg_compression"
]

print("Corruption types:", corruption_types)
print("Severity levels: 1 (mild), 2 (medium), 3 (severe)")

In [ ]:
# Cell 83: Preview the five corruption types

sample_path = "/content/" + manifest_df.iloc[0]["image_path"]
original_image = Image.open(sample_path).convert("RGB")

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 6, figsize=(18, 4))

axes[0].imshow(original_image)
axes[0].set_title("Original")

for i, corruption in enumerate(corruption_types):
    corrupted = apply_corruption(
        original_image, corruption, severity=2, seed=42
    )
    axes[i + 1].imshow(corrupted)
    axes[i + 1].set_title(corruption.replace("_", "\n"))

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 84: Check that corruption outputs are reproducible

test_image = np.array(original_image)

result_1 = apply_corruption(
    test_image, "gaussian_noise", severity=2, seed=42
)

result_2 = apply_corruption(
    test_image, "gaussian_noise", severity=2, seed=42
)

result_3 = apply_corruption(
    test_image, "gaussian_noise", severity=2, seed=99
)

print("Same seed produces identical output:",
      np.array_equal(result_1, result_2))

print("Different seed produces different output:",
      not np.array_equal(result_1, result_3))

In [ ]:
# Cell 85: Test all corruption types and severity levels

test_image = np.array(original_image)

for corruption in corruption_types:
    for severity in [1, 2, 3]:
        result = apply_corruption(
            test_image, corruption, severity, seed=42
        )

        assert result.shape == test_image.shape
        assert result.dtype == np.uint8

        print(f"Passed: {corruption}, severity {severity}")

print("\nAll 15 corruption settings passed!")

In [ ]:
# Cell 86: Save corruption configuration

import json

corruption_config = {
    "corruption_types": corruption_types,
    "severity_levels": [1, 2, 3],
    "random_seed": 42,
    "description": "Five image corruptions with three severity levels."
}

config_path = "/content/data/corruptions/corruption_config.json"

with open(config_path, "w") as file:
    json.dump(corruption_config, file, indent=4)

print("Configuration saved:", config_path)
print("Total corruption settings:",
      len(corruption_types) * 3)

In [ ]:
# Cell 87: Create frozen test split with SHA-256 checksums

import hashlib

test_df = manifest_df[
    manifest_df["benchmark_split"] == "test"
].copy()

def calculate_sha256(image_path):
    full_path = "/content/" + image_path

    with open(full_path, "rb") as file:
        return hashlib.sha256(file.read()).hexdigest()

test_df["sha256"] = test_df["image_path"].apply(calculate_sha256)

frozen_path = "/content/data/frozen_test_manifest.csv"
test_df.to_csv(frozen_path, index=False)

# Create one overall checksum for the frozen test split
combined_hash = hashlib.sha256(
    "".join(
        test_df.sort_values("image_id")["sha256"].tolist()
    ).encode()
).hexdigest()

with open("/content/data/test_split_sha256.txt", "w") as file:
    file.write(combined_hash)

print("Frozen test images:", len(test_df))
print("Missing checksums:", test_df["sha256"].isna().sum())
print("Frozen manifest saved:", frozen_path)
print("Overall SHA-256:", combined_hash)

In [ ]:
# Cell 88: Verify frozen test split

saved_test_df = pd.read_csv(
    "/content/data/frozen_test_manifest.csv"
)

recalculated_hashes = saved_test_df["image_path"].apply(
    calculate_sha256
)

hashes_match = (
    recalculated_hashes.tolist()
    == saved_test_df["sha256"].tolist()
)

new_combined_hash = hashlib.sha256(
    "".join(
        saved_test_df.sort_values("image_id")["sha256"].tolist()
    ).encode()
).hexdigest()

with open("/content/data/test_split_sha256.txt") as file:
    saved_combined_hash = file.read().strip()

print("Image checksums match:", hashes_match)
print("Overall checksum matches:", new_combined_hash == saved_combined_hash)
print("Frozen test image count:", len(saved_test_df))

In [ ]:
# Cell 89: Check available images for the contact sheet

print("Total unique images:", manifest_df["image_id"].nunique())

print("\nImages by benchmark split:")
print(manifest_df["benchmark_split"].value_counts())

print("\nAvailable English labels:")
print(
    sorted(set(
        label
        for labels in manifest_df["english_labels"].dropna()
        for label in labels.split(";")
    ))
)

print("\nTaxonomy Mandarin labels available:",
      "mandarin_name" in taxonomy_df.columns)

In [ ]:
# Cell 90: Select contact sheet samples

sample_df = (
    manifest_df[
        manifest_df["benchmark_split"].isin(["test", "validation"])
    ]
    .sort_values(
        ["benchmark_split", "image_id"],
        ascending=[True, True]
    )
    .groupby("english_labels", group_keys=False)
    .head(2)
    .copy()
)

print("Selected sample images:", len(sample_df))
print("Represented classes:", sample_df["english_labels"].nunique())
print("\nImages per class:")
print(sample_df["english_labels"].value_counts())

In [ ]:
# Cell 91: Select samples by individual class

# Separate images that have multiple labels
sample_candidates = manifest_df.copy()
sample_candidates["english_label"] = (
    sample_candidates["english_labels"].str.split(";")
)

sample_candidates = sample_candidates.explode("english_label")

# Select up to two images per individual class
sample_df = (
    sample_candidates
    .sort_values("benchmark_split")
    .groupby("english_label")
    .head(2)
    .drop_duplicates(subset="image_id")
    .copy()
)

# Add Mandarin labels from the taxonomy
sample_df = sample_df.merge(
    taxonomy_df[["english_name", "mandarin_name"]],
    left_on="english_label",
    right_on="english_name",
    how="left"
)

print("Selected unique images:", sample_df["image_id"].nunique())
print("Individual classes represented:", sample_df["english_label"].nunique())
print("Missing Mandarin labels:", sample_df["mandarin_name"].isna().sum())

In [ ]:
# Cell 92: Add one image to reach the 40-image minimum

remaining_candidates = sample_candidates[
    ~sample_candidates["image_id"].isin(sample_df["image_id"])
]

extra_sample = remaining_candidates.iloc[[0]].copy()

extra_sample = extra_sample.merge(
    taxonomy_df[["english_name", "mandarin_name"]],
    left_on="english_label",
    right_on="english_name",
    how="left"
)

sample_df = pd.concat(
    [sample_df, extra_sample],
    ignore_index=True
)

print("Total unique sample images:", sample_df["image_id"].nunique())
print("Missing Mandarin labels:", sample_df["mandarin_name"].isna().sum())

In [ ]:
# Cell 93: Generate bilingual contact sheet

import matplotlib.pyplot as plt
from PIL import Image
import math

sample_df = sample_df.reset_index(drop=True)

columns = 5
rows = math.ceil(len(sample_df) / columns)

fig, axes = plt.subplots(rows, columns, figsize=(16, rows * 3))

for ax, (_, row) in zip(axes.flat, sample_df.iterrows()):
    image_path = "/content/" + row["image_path"]
    image = Image.open(image_path).convert("RGB")

    ax.imshow(image)
    ax.set_title(
        f'{row["english_label"]}\n{row["mandarin_name"]}',
        fontsize=9
    )
    ax.axis("off")

for ax in axes.flat[len(sample_df):]:
    ax.axis("off")

plt.tight_layout()

contact_sheet_path = "/content/data/W03_Benchmark_Samples.png"
plt.savefig(contact_sheet_path, dpi=200, bbox_inches="tight")
plt.show()

print("Contact sheet saved:", contact_sheet_path)
print("Images included:", len(sample_df))

In [ ]:
# Cell 94: Fix Mandarin font rendering

!apt-get -qq install fonts-noto-cjk

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
chinese_font = fm.FontProperties(fname=font_path)

fig, axes = plt.subplots(8, 5, figsize=(16, 24))

for ax, (_, row) in zip(axes.flat, sample_df.iterrows()):
    image_path = "/content/" + row["image_path"]
    image = Image.open(image_path).convert("RGB")

    ax.imshow(image)
    ax.set_title(
        f'{row["english_label"]}\n{row["mandarin_name"]}',
        fontproperties=chinese_font,
        fontsize=10
    )
    ax.axis("off")

for ax in axes.flat[len(sample_df):]:
    ax.axis("off")

plt.tight_layout()

contact_sheet_path = "/content/data/W03_Benchmark_Samples.png"
plt.savefig(contact_sheet_path, dpi=200, bbox_inches="tight")
plt.show()

print("Updated bilingual contact sheet:", contact_sheet_path)

In [ ]:
# Cell 95: Find images with multiple labels

multilabel_df = manifest_df[
    manifest_df["english_labels"].str.contains(";", na=False)
].copy()

print("Images with multiple labels:", len(multilabel_df))

display(
    multilabel_df[
        ["image_id", "english_labels", "image_path"]
    ].head(15)
)

In [ ]:
# Cell 96: Display 15 potential hard cases

import matplotlib.pyplot as plt
from PIL import Image

candidates = multilabel_df.head(15).reset_index(drop=True)

fig, axes = plt.subplots(3, 5, figsize=(16, 10))

for ax, (_, row) in zip(axes.flat, candidates.iterrows()):
    image_path = "/content/" + row["image_path"]
    image = Image.open(image_path).convert("RGB")

    ax.imshow(image)
    ax.set_title(row["english_labels"], fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 97: Select and save five hard cases

hard_case_ids = [
    "20406256853c0e5c",  # Fork; Spoon
    "2b5a92af4b57d139",  # Door; Window
    "361742ff1a64af93",  # Stairs; Window
    "3b3b9931325d65c9",  # Book; Desk
    "2a0905a2ff63f09e"   # Chair; Desk; Table
]

hard_cases_df = manifest_df[
    manifest_df["image_id"].isin(hard_case_ids)
].copy()

hard_cases_df.to_csv(
    "/content/data/hard_cases.csv",
    index=False
)

print("Hard cases selected:", len(hard_cases_df))
display(hard_cases_df[["image_id", "english_labels"]])

In [ ]:
# Cell 98: Combine regular samples and hard cases

hard_case_samples = hard_cases_df.copy()

# Create bilingual labels for images with multiple objects
mandarin_lookup = dict(zip(
    taxonomy_df["english_name"],
    taxonomy_df["mandarin_name"]
))

hard_case_samples["english_label"] = hard_case_samples["english_labels"]
hard_case_samples["mandarin_name"] = hard_case_samples["english_labels"].apply(
    lambda labels: " / ".join(
        mandarin_lookup[label] for label in labels.split(";")
    )
)

hard_case_samples["hard_case"] = True
sample_df["hard_case"] = sample_df["image_id"].isin(hard_case_ids)

# Combine and remove duplicate images
final_samples_df = pd.concat(
    [sample_df, hard_case_samples],
    ignore_index=True
).drop_duplicates(subset="image_id")

print("Final unique images:", final_samples_df["image_id"].nunique())
print("Hard cases included:", final_samples_df["hard_case"].sum())

In [ ]:
# Cell 99: Generate final bilingual contact sheet

import matplotlib.pyplot as plt
from PIL import Image
import math

final_samples_df = final_samples_df.reset_index(drop=True)

columns = 5
rows = math.ceil(len(final_samples_df) / columns)

fig, axes = plt.subplots(rows, columns, figsize=(18, rows * 3.5))

for ax, (_, row) in zip(axes.flat, final_samples_df.iterrows()):
    image_path = "/content/" + row["image_path"]
    image = Image.open(image_path).convert("RGB")

    ax.imshow(image)

    label = f'{row["english_label"]}\n{row["mandarin_name"]}'

    if row["hard_case"]:
        label = "[HARD CASE]\n" + label

    ax.set_title(
        label,
        fontproperties=chinese_font,
        fontsize=9
    )
    ax.axis("off")

for ax in axes.flat[len(final_samples_df):]:
    ax.axis("off")

plt.tight_layout()

final_sheet_path = "/content/data/W03_Benchmark_Samples.png"
plt.savefig(final_sheet_path, dpi=200, bbox_inches="tight")
plt.show()

# Save the sample index too
final_samples_df.to_csv(
    "/content/data/W03_Benchmark_Samples_Index.csv",
    index=False
)

print("Final contact sheet saved:", final_sheet_path)
print("Sample index saved.")
print("Total images:", len(final_samples_df))
print("Hard cases:", final_samples_df["hard_case"].sum())

In [ ]:
# Cell 100: Prepare blind label audit

audit_sample = manifest_df.sample(
    n=57,
    random_state=42
).copy()

# Reviewer sheet: hide original labels
reviewer_sheet = audit_sample[
    ["image_id", "image_path"]
].copy()

reviewer_sheet["blind_relabel"] = ""
reviewer_sheet["review_date"] = ""
reviewer_sheet["reviewer_notes"] = ""

reviewer_sheet.to_csv(
    "/content/data/label_audit_blind.csv",
    index=False
)

# Private answer key: preserve original labels separately
audit_answer_key = audit_sample[
    ["image_id", "english_labels"]
].copy()

audit_answer_key.to_csv(
    "/content/data/label_audit_answer_key.csv",
    index=False
)

print("Images selected for blind audit:", len(reviewer_sheet))
print("Original labels hidden from reviewer.")
print("Blind audit sheet saved.")
print("Answer key saved separately.")

In [ ]:
# Cell 101: Prepare independent second-reader review

second_reader_sheet = reviewer_sheet.sample(
    n=50,
    random_state=43
).copy()

second_reader_sheet["second_reader_label"] = ""
second_reader_sheet["reviewer_notes"] = ""

second_reader_sheet.to_csv(
    "/content/data/second_reader_review.csv",
    index=False
)

# Save the audit procedure
audit_instructions = """
# Week 3 Label Audit

## Blind relabeling
- Randomly selected 57 images (approximately 10% of the dataset).
- Original labels are stored separately in label_audit_answer_key.csv.
- A reviewer should independently relabel the images without seeing the original labels.
- Compare the new labels with the original labels after review.

## Second-reader review
- A separate sample of 50 images is provided in second_reader_review.csv.
- A second reader should independently label these images.
- Compare disagreements and document how they are resolved.

## Status
The review sheets have been prepared, but the actual reviews,
agreement calculations, and Cohen's kappa are pending.
"""

with open("/content/data/W03_Label_Audit.md", "w") as file:
    file.write(audit_instructions)

print("Second-reader images:", len(second_reader_sheet))
print("Audit instructions saved.")
print("Actual reviews and Cohen's kappa: pending")

In [ ]:
# Cell 103: Final Week 3 file verification

import os

required_files = [
    "/content/data/manifest.csv",
    "/content/data/split_summary.csv",
    "/content/data/frozen_test_manifest.csv",
    "/content/data/test_split_sha256.txt",
    "/content/data/corruptions/corruption_config.json",
    "/content/data/W03_Benchmark_Samples.png",
    "/content/data/W03_Benchmark_Samples_Index.csv",
    "/content/data/hard_cases.csv",
    "/content/data/W03_Label_Audit.md",
    "/content/data/label_audit_blind.csv",
    "/content/data/label_audit_answer_key.csv",
    "/content/data/second_reader_review.csv"
]

for path in required_files:
    status = "FOUND" if os.path.exists(path) else "MISSING"
    print(status, os.path.basename(path))

print("\nTotal images:", len(manifest_df))
print("Test images:", len(test_df))
print("Sample images:", len(final_samples_df))